## Statistics: Confidence Intervals
###### A confidence interval (CI) is a range of values calculated from a sample that likely contains the true population parameter. If we repeatedly drew samples from the population and built a CI each time, a certain percentage (e.g., 95%) of those intervals would include the actual population value.

In [1]:
import numpy as np
from scipy.stats import norm, t

In [2]:
np.random.seed(1)

### Generate sample from distribution
###### X - will be sample

In [3]:
N=1000
mu=5
sigma=2
X=np.random.randn(N)*sigma + mu

### Confidence Interval - Theory
###### For a random variable $X \sim N(\mu, \sigma^2)$, the critical value $z_{\alpha/2}$ for a two-sided confidence interval can be found using the integral:

$$
\frac{\alpha}{2} = P(X \le z_{\alpha/2}) = \int_{-\infty}^{z_{\alpha/2}} f_X(x) \, dx
$$

###### where the **probability density function** `PDF` is:

$$
f_X(x) = \frac{1}{\sigma \sqrt{2\pi}} \exp\Big(-\frac{(x-\mu)^2}{2\sigma^2}\Big)
$$

* ###### The integral of the pdf from $-\infty$ to $z_{\alpha/2}$ gives the probability in the left tail, e.g., 0.025 for a 95% confidence interval.
* ###### Symmetrically, the right tail satisfies $P(X \ge z_{1-\alpha/2}) = \alpha/2$.

###### In practice, we often use the **percent point function** `PPF` to get $z_{\alpha/2}$ quickly:

$$
z_{\alpha/2} = \Phi^{-1}(\alpha/2)
$$

###### This gives the numerical critical value used to construct the confidence interval.
### Z-Confidence Interval vs T-Confidence Interval
###### In the context of **confidence intervals**, the **z-statistic** and **t-statistic** are both measures of how far the sample mean is from the population mean in standardized units, but they are used under different assumptions:

###### Z-statistic

- ###### Defined as:

$$
z = \frac{\bar{X} - \mu}{\sigma / \sqrt{n}}
$$

- ###### Measures the deviation of the sample mean from the population mean in units of the population standard error.
- ###### Assumes **the population variance $\sigma^2$ is known** or the sample size is large (so the Central Limit Theorem makes the approximation valid).
- ###### For a given confidence level, the **critical z-value** determines the width of the confidence interval:

$$
\bar{X} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}
$$

---

###### T-statistic

- ###### Defined as:

$$
t = \frac{\bar{X} - \mu}{s / \sqrt{n}}
$$

- ###### Similar in form to the z-statistic, but uses the **sample standard deviation $s$** instead of $\sigma$.
- ###### Accounts for the extra uncertainty when the population variance is unknown.
- ###### The t-statistic follows a **Student’s t-distribution** with $n-1$ degrees of freedom, which has thicker tails than the normal distribution, making the CI slightly wider for small samples:

$$
\bar{X} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}
$$

---

###### Summary

- ###### Both statistics are central to **constructing confidence intervals**.
- ###### The choice between z and t depends on **whether the population variance is known** and the **sample size**.
- ###### They quantify the uncertainty of the sample mean estimate and determine **how far the CI extends from the observed sample mean**, reflecting the precision of the estimate.
#### Z-Confidence Interval

In [4]:
mu_hat = np.mean(X)
sigma_hat = np.std(X, ddof=1)
# We need 'z critical (left and right)', it corresponds to the 'significance level α = 0.95 - 95%' -> 'α/2', which we already know because we set it earlier.
z_left = norm.ppf(0.025)
z_right = norm.ppf(0.975)
lower = mu_hat + z_left * sigma_hat / np.sqrt(N)
upper = mu_hat + z_right * sigma_hat / np.sqrt(N)
print(mu_hat, lower, upper)

5.077624952319204 4.955959806754385 5.199290097884023


#### T-Confidence Interval

In [5]:
mu_hat = np.mean(X)
sigma_hat = np.std(X, ddof=1)
t_left = t.ppf(0.025, df = N - 1)
t_right = t.ppf(0.975, df = N - 1)
lower = mu_hat + t_left * sigma_hat / np.sqrt(N)
upper = mu_hat + t_right * sigma_hat / np.sqrt(N)
print(mu_hat, lower, upper)

5.077624952319204 4.9558122244324165 5.199437680205992


### Confidence Interval Coverage Simulation (Monte Carlo)

###### The purpose of this experiment is to **empirically verify** the theoretical coverage probability of the **95% confidence interval** for the population mean ($\mu$).

* ###### The `experiment()` function: Simulates taking a single random sample ($X$) and calculating its confidence interval. It returns `True` (1) if the interval **successfully covers** the true mean $\mu$, and `False` (0) otherwise.
* ###### The `multi_experiment(M)` function: Repeats this process **$M$ times** and calculates the mean of the results, which yields the **percentage of successful covers**.

###### The **final result** should be approximately **0.95**, demonstrating that the calculated confidence intervals work as designed.


In [6]:
def experiment():
  X = np.random.randn(N)*sigma + mu
  mu_hat = np.mean(X)
  sigma_hat = np.std(X, ddof=1)
  t_left = t.ppf(0.025, df=N - 1)
  t_right = t.ppf(0.975, df=N - 1)
  lower = mu_hat + t_left * sigma_hat / np.sqrt(N)
  upper = mu_hat + t_right * sigma_hat / np.sqrt(N)
  return mu > lower and mu < upper

In [7]:
def multi_experiment(M):
  results = [experiment() for _ in range(M)]
  return np.mean(results)

In [8]:
print(multi_experiment(10000))

0.9506
